# Laboratory 3: Image Features

Part B loads images from the workspace `lab3/cats` and `lab3/Dogs` folders, converts them to grayscale, resizes them to a consistent size, and flattens the pixels into feature vectors.

In [3]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

cwd = Path.cwd()
possible_roots = [cwd, cwd / "lab3", cwd.parent / "lab3"]
IMAGE_ROOT = next((root for root in possible_roots if (root / "cats").is_dir()), None)
if IMAGE_ROOT is None:
    raise FileNotFoundError("Could not find the lab3 folder containing cats/.")
CAT_DIR = IMAGE_ROOT / "cats"
DOG_DIR = IMAGE_ROOT / "Dogs"
if not DOG_DIR.is_dir():
    DOG_DIR = IMAGE_ROOT / "dogs"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
TARGET_SIZE = (64, 64)


def image_to_features(path, size=TARGET_SIZE):
    with Image.open(path) as image:
        grayscale = image.convert("L").resize(size)
        return np.asarray(grayscale, dtype=np.uint8).flatten()

In [ ]:
cat_images = sorted(path for path in CAT_DIR.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
if not cat_images:
    raise FileNotFoundError(f"No supported cat images found in {CAT_DIR}")
image_path = cat_images[0]
with Image.open(image_path) as source_image:
    original = source_image.convert("L")
    resized_64 = original.resize((64, 64))
    resized_32 = original.resize((32, 32))

arrays = [np.asarray(original), np.asarray(resized_64), np.asarray(resized_32)]
titles = [
    f"Original {arrays[0].shape} ({arrays[0].size:,} pixels)",
    f"64x64 ({arrays[1].size:,} pixels)",
    f"32x32 ({arrays[2].size:,} pixels)",
]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, array, title in zip(axes, arrays, titles):
    ax.imshow(array, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(image_path.name)
plt.tight_layout()
plt.show()

In [ ]:
def collect_class(folder, label):
    samples = []
    for path in sorted(folder.iterdir()):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            samples.append((image_to_features(path), label, path.name))
    return samples

samples = collect_class(CAT_DIR, "cat") + collect_class(DOG_DIR, "dog")
if not samples:
    raise FileNotFoundError(f"No images found under {CAT_DIR} and {DOG_DIR}")
X = np.stack([sample[0] for sample in samples]).astype(np.float32) / 255.0
y = np.asarray([sample[1] for sample in samples])
filenames = np.asarray([sample[2] for sample in samples])
OUTPUT_DIR = IMAGE_ROOT.parent / "lab03"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.savez(OUTPUT_DIR / "dataset.npz", X=X, y=y, filenames=filenames)
print(f"Cat images: {sum(label == 'cat' for label in y)}")
print(f"Dog images: {sum(label == 'dog' for label in y)}")
print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Saved dataset to {OUTPUT_DIR / 'dataset.npz'}")

## Discussion

1. Grayscale conversion reduces three color channels to one luminance value per pixel, discarding hue and saturation.
2. An $H \times W$ grayscale image becomes a vector of length $H \times W$.
3. Flattening represents the image as a one-dimensional vector accepted by standard tabular models, but loses the explicit two-dimensional neighborhood structure.
4. Resizing gives every image the same dimensions and therefore the same feature-vector length.